# Data Cleaning and Database Normalization
Cleaning raw data, handling missing values, and exporting 6 normalized relational tables.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load Raw Data and Remove Empty Padding

In [2]:
df_raw = pd.read_csv('../data/raw/Pakistan Largest Ecommerce Dataset.csv', low_memory=False)
df = df_raw.dropna(how='all')
unnamed = [c for c in df.columns if 'Unnamed' in c]
df = df.drop(columns=unnamed)
print("Cleaned shape:", df.shape)

Cleaned shape: (584524, 21)


## 2. Clean Text Fields and Fill Missing Values

In [3]:
df['category_name_1'] = df['category_name_1'].replace({r'\N': np.nan, '\\N': np.nan}).fillna('Others').astype(str).str.strip()
df['payment_method'] = df['payment_method'].astype(str).str.lower().str.strip()
df['payment_method'] = df['payment_method'].replace({'\\n': 'unknown', 'nan': 'unknown', '': 'unknown'})
df['status'] = df['status'].astype(str).str.lower().str.strip()
df['status'] = df['status'].replace({'nan': 'unknown', '': 'unknown'})
df['sku'] = df['sku'].fillna('UNKNOWN_SKU').astype(str).str.strip()

## 3. Convert Data Types and Format Dates

In [4]:
df['Customer ID'] = pd.to_numeric(df['Customer ID'], errors='coerce').fillna(0).astype(int)
df['Customer Since'] = pd.to_datetime(df['Customer Since'], errors='coerce').dt.strftime('%Y-%m-%d')
df['order_date'] = pd.to_datetime(df['created_at'], errors='coerce')
df['order_date'] = df['order_date'].fillna(pd.to_datetime(df['Working Date'], errors='coerce')).dt.strftime('%Y-%m-%d')

df['item_id'] = pd.to_numeric(df['item_id'], errors='coerce').astype(np.int64)
df['increment_id'] = df['increment_id'].astype(str).str.strip()
df['qty_ordered'] = pd.to_numeric(df['qty_ordered'], errors='coerce').fillna(1).astype(int)
df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(0.0).round(2)
df['discount_amount'] = pd.to_numeric(df['discount_amount'], errors='coerce').fillna(0.0).abs().round(2)
df['grand_total'] = pd.to_numeric(df['grand_total'], errors='coerce').fillna(0.0).round(2)

## 4. Create Master Tables (Categories, PaymentMethods, Products, Customers)

In [5]:
unique_cats = sorted(df['category_name_1'].unique())
categories = pd.DataFrame({
    'category_id': range(1, len(unique_cats) + 1),
    'category_name': unique_cats
})
cat_map = dict(zip(categories['category_name'], categories['category_id']))
df['category_id'] = df['category_name_1'].map(cat_map)

unique_pm = sorted(df['payment_method'].unique())
payment_methods = pd.DataFrame({
    'payment_method_id': range(1, len(unique_pm) + 1),
    'payment_method_name': unique_pm
})
pm_map = dict(zip(payment_methods['payment_method_name'], payment_methods['payment_method_id']))
df['payment_method_id'] = df['payment_method'].map(pm_map)

prod_agg = df.groupby('sku').agg({'category_id': 'first', 'price': 'mean'}).reset_index()
products = pd.DataFrame({
    'product_id': range(1, len(prod_agg) + 1),
    'sku': prod_agg['sku'],
    'category_id': prod_agg['category_id'].astype(int),
    'price': prod_agg['price'].round(2)
})
prod_map = dict(zip(products['sku'], products['product_id']))
df['product_id'] = df['sku'].map(prod_map)

cust_agg = df.groupby('Customer ID')['Customer Since'].first().reset_index()
customers = pd.DataFrame({
    'customer_id': cust_agg['Customer ID'],
    'customer_since': cust_agg['Customer Since']
})

print("Categories count:", len(categories))
print("PaymentMethods count:", len(payment_methods))
print("Products count:", len(products))
print("Customers count:", len(customers))

Categories count: 15
PaymentMethods count: 18
Products count: 84800
Customers count: 115327


## 5. Create Transaction Tables (Orders and OrderDetails)

In [6]:
order_agg = df.groupby('increment_id').agg({
    'Customer ID': 'first',
    'order_date': 'first',
    'payment_method_id': 'first',
    'status': 'first',
    'grand_total': 'first'
}).reset_index()

orders = pd.DataFrame({
    'order_id': order_agg['increment_id'],
    'customer_id': order_agg['Customer ID'].astype(int),
    'order_date': order_agg['order_date'],
    'payment_method_id': order_agg['payment_method_id'].astype(int),
    'status': order_agg['status'],
    'grand_total': order_agg['grand_total'].round(2)
})

order_details = pd.DataFrame({
    'order_detail_id': df['item_id'],
    'order_id': df['increment_id'],
    'product_id': df['product_id'].astype(int),
    'qty_ordered': df['qty_ordered'],
    'price': df['price'].round(2),
    'discount_amount': df['discount_amount'].round(2),
    'total_amount': ((df['qty_ordered'] * df['price']) - df['discount_amount']).round(2)
})

print("Orders count:", len(orders))
print("OrderDetails count:", len(order_details))

Orders count: 408782
OrderDetails count: 584524


## 6. Validate Primary Keys and Foreign Key Relationships

In [7]:
print("Orders PK unique:", orders['order_id'].is_unique)
print("OrderDetails PK unique:", order_details['order_detail_id'].is_unique)
print("Products PK unique:", products['product_id'].is_unique)
print("Customers PK unique:", customers['customer_id'].is_unique)

print("Orphan customers in orders:", (~orders['customer_id'].isin(customers['customer_id'])).sum())
print("Orphan orders in details:", (~order_details['order_id'].isin(orders['order_id'])).sum())
print("Orphan products in details:", (~order_details['product_id'].isin(products['product_id'])).sum())

Orders PK unique: True
OrderDetails PK unique: True
Products PK unique: True
Customers PK unique: True
Orphan customers in orders: 0


Orphan orders in details: 0
Orphan products in details: 0


## 7. Export Clean CSV Files

In [8]:
output_dir = '../data/cleaned'
os.makedirs(output_dir, exist_ok=True)

categories.to_csv(f'{output_dir}/Categories.csv', index=False)
payment_methods.to_csv(f'{output_dir}/PaymentMethods.csv', index=False)
products.to_csv(f'{output_dir}/Products.csv', index=False)
customers.to_csv(f'{output_dir}/Customers.csv', index=False)
orders.to_csv(f'{output_dir}/Orders.csv', index=False)
order_details.to_csv(f'{output_dir}/OrderDetails.csv', index=False)

print("All 6 relational tables successfully saved to data/cleaned!")

All 6 relational tables successfully saved to data/cleaned!
